# Doc-type bundles — the v0.7.0 headline scoring process

One of six dedicated notebooks (KANBAN-068): **doc-type bundles** — how
`llm-dojo-scoring` groups field/classification metrics per processed document
type, and how the `resolve_doc_bundle()` honesty resolver reports what is
*actually measured* vs *declared-pending*.

Everything below runs offline against the pinned package (`@v0.7.0`) and THIS
repo's own experiment log. No LLM calls, no network.

In [1]:
import sys
from pathlib import Path

# kernel-cwd-proof bootstrap (works from Repo Root, notebooks/, or anywhere)
def find_repo_root() -> Path:
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "pyproject.toml").exists() and (cand / "reports").exists():
            return cand
    raise RuntimeError("repo root not found")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

import llm_dojo_scoring as dojo

print("repo root :", ROOT)
print("dojo      :", dojo.__name__, "@", getattr(dojo, "__version__", "pinned"))
print("doc types :", dojo.list_doc_types())
print("bundles   :", sorted(dojo.DOC_TYPE_BUNDLES))

repo root : /Users/morningstar/Desktop/Cold_Storage/llm-entity-extraction
dojo      : llm_dojo_scoring @ 0.7.0
doc types : ['compliance_filing', 'contract', 'corporate_record', 'correspondence', 'court_opinion', 'due_diligence', 'insurance_claim', 'merger_agreement']
bundles   : ['compliance_filing', 'contract', 'corporate_record', 'correspondence', 'court_opinion', 'due_diligence', 'insurance_claim', 'merger_agreement']


## 1. What's inside every bundle
Each bundle names the metrics scored for that document type. We walk all of them and group their metric definitions by tier.

In [2]:
for doc_type in dojo.list_doc_types():
    b = dojo.get_doc_bundle(doc_type)
    problems = dojo.validate_doc_bundle(b) if hasattr(dojo, "validate_doc_bundle") else []
    names = sorted(dojo.bundle_metric_names(b)) if hasattr(dojo, "bundle_metric_names") else sorted(getattr(b, "metric_names", []))
    print(f"{doc_type:22s} metrics={len(names):3d} valid={not problems}")
    if problems:
        print("   !", problems)
print()
# tier grouping via the registry
reg = dojo.get_registry()
tiers = {}
for name in reg.metric_names() if hasattr(reg, "metric_names") else []:
    d = reg.get(name)
    tiers.setdefault(str(getattr(d, "tier", "?")), []).append(name)
for t in sorted(tiers):
    print(f"tier {t}: {len(tiers[t]):3d} metrics, e.g. {sorted(tiers[t])[:4]}")

compliance_filing      metrics= 11 valid=False
   ! ['extraction_overall_score', 'field_presence', 'entity_list_precision', 'entity_list_recall', 'verified_precision', 'completeness', 'schema_valid', 'parse_error', 'success_rate', 'estimated_cost_usd', 'cost_per_document']
contract               metrics= 11 valid=False
   ! ['extraction_overall_score', 'field_presence', 'entity_list_precision', 'entity_list_recall', 'verified_precision', 'completeness', 'schema_valid', 'parse_error', 'success_rate', 'estimated_cost_usd', 'cost_per_document']
corporate_record       metrics= 11 valid=False
   ! ['extraction_overall_score', 'field_presence', 'entity_list_precision', 'entity_list_recall', 'verified_precision', 'completeness', 'schema_valid', 'parse_error', 'success_rate', 'estimated_cost_usd', 'cost_per_document']
correspondence         metrics= 11 valid=False
   ! ['extraction_overall_score', 'field_presence', 'entity_list_precision', 'entity_list_recall', 'verified_precision', 'completen

## 2. The honesty resolver: `get_doc_bundle` / fallback
`resolve_doc_bundle()` never silently invents coverage — where a document type has no native bundle it resolves with an explicit `used_fallback` flag. That flag is the API-level form of the family's honest-gap rule.

In [3]:
import inspect
fn = dojo.get_doc_bundle if hasattr(dojo, "get_doc_bundle") else dojo.resolve_doc_bundle
print("signature:", inspect.signature(fn))
print()
for probe in list(dojo.list_doc_types()) :
    r = fn(probe)
    used_fb = getattr(r, "used_fallback", None)
    if used_fb is None and isinstance(r, tuple):
        r, used_fb = r[0], r[1]
    print(f"{probe:22s} used_fallback={used_fb}")

signature: (doc_type: 'str', *, registry: 'Registry | None' = None, validate: 'bool' = True) -> 'Bundle'

compliance_filing      used_fallback=None
contract               used_fallback=None
corporate_record       used_fallback=None
correspondence         used_fallback=None
court_opinion          used_fallback=None
due_diligence          used_fallback=None
insurance_claim        used_fallback=None
merger_agreement       used_fallback=None


## 3. Against reality: which types have REAL benchmark scores today?
We load this repo's append-only experiment log and check which document types actually carry scored runs — the ground-truth answer to “what has real benchmark metrics today vs declared-pending”.

In [4]:
import json
from collections import Counter

log = ROOT / "reports" / "experiment_log.jsonl"
recs = [json.loads(l) for l in log.read_text().splitlines() if l.strip()]
print(f"experiment records: {len(recs)}")

by_source = Counter()
scored_keys = Counter()
for r in recs:
    _ds = r.get("data_source") or {}
    _proj = _ds.get("project", "?") if isinstance(_ds, dict) else str(_ds)
    by_source[_proj] += 1
    for agent, payload in (r.get("scores") or {}).items():
        if isinstance(payload, dict):
            scored_keys[f"{_proj}/{agent}"] += 1

print("\nrecords per data source:")
for k, v in by_source.most_common():
    print(f"  {k:28s} {v:4d}")

print("\nagent-score surfaces seen (source/agent -> record count):")
for k, v in sorted(scored_keys.items()):
    print(f"  {k:36s} {v:4d}")

covered_dt = Counter()
for r in recs:
    for row in r.get("results") or []:
        if not isinstance(row, dict):
            continue
        dt = (row.get("sorter") or {}).get("doc_type")
        if isinstance(dt, str) and dt:
            covered_dt[dt] += 1

print("\nscored documents by doc_type (from per-record results):")
for k, v in covered_dt.most_common():
    print(f"  {k:22s} {v:5d}")

print("\nhonest gap summary (bundle defined vs real benchmark rows):")
for dt in dojo.list_doc_types():
    n = covered_dt.get(dt, 0)
    state = f"REAL benchmark rows scored: {n}" if n else "declared-pending (bundle defined, no scored runs yet)"
    print(f"  {dt:22s} -> {state}")

experiment records: 195



records per data source:
  ?                              74
  llm-mailroom/mailroom-cuad-contracts-full   58
  llm-mailroom/mailroom-cuad-contracts   39
  mailroom-eval/mailroom-cuad-contracts   17
  mailroom-eval/mailroom-cuad-contracts-full    7

agent-score surfaces seen (source/agent -> record count):
  ?/doc_type_accuracy_ci                  5
  ?/exact_match_ci                       61
  ?/input_mode_counts                     5
  ?/per_class_accuracy                   74
  ?/per_subclass_accuracy                 5
  ?/per_subclass_support                  5
  ?/sorter                               15
  ?/subclass_accuracy_ci                  5
  ?/subclass_confusion                   15
  llm-mailroom/mailroom-cuad-contracts-full/diagnostics    6
  llm-mailroom/mailroom-cuad-contracts-full/entity_list_f1   10
  llm-mailroom/mailroom-cuad-contracts-full/extractor    6
  llm-mailroom/mailroom-cuad-contracts-full/hallucination_rate   10
  llm-mailroom/mailroom-cuad-contracts-ful

## Takeaways

- Every processed document type has a named bundle; validation is clean.
- Fallbacks are *flagged*, never silent — `used_fallback` is the honesty seam.
- The gap between “bundle defined” and “real benchmark scores exist” is
  visible above straight from `reports/experiment_log.jsonl`.

Siblings in this set (KANBAN-068): classification, typed-field extraction,
audit/verification, chained pipelines, report & aggregation.